# Teks-only vs Metadata: apakah metadata mengubah prediksi?

Teks yang **sama** dimasukkan ke dua model:

- **Model teks-only**: satu prediksi per teks. Model ini dilatih dengan label _majority vote_, jadi prediksinya mewakili pola mayoritas.
- **Model metadata**: satu prediksi per (teks, annotator), karena metadata annotator ikut masuk ke input.

Pertanyaannya: apakah model metadata **ikut pola teks-only** (prediksinya sama untuk semua annotator), atau **menangkap metadata** (prediksinya berubah kalau annotator-nya berbeda)?

Dicek untuk `toxicity` (stage 1) dan kelima jenis toxicity (stage 2).


In [11]:
! pip install -q emoji

In [12]:
import os
import sys
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# preprocess.py: di Colab upload ke /content, lokal ambil dari reusable-scripts
for p in ["/content", "../../reusable-scripts"]:
    sys.path.append(p)
from preprocess import preprocess_text

## Konfigurasi
Ganti `CASES_PATH` untuk memilih testcase:
- `testcases_text_metadata.csv` → format pasangan (teks, annotator), Stage 1
- `testcases_toxicity_type.csv` → cross join semua annotator, Stage 2 (jenis toxicity)
- `testcases_disagreement.csv` → cross join semua annotator, Stage 1

In [26]:
CASES_PATH = "testcases_text_metadata.csv"
USE_ALL_ANNOTATORS = False
PROFILES_PATH = "../data/explode_dataset_using_latest_cleaned_text.csv"
OUT_DIR = "results"

TEXT_S1 = "evelnap/indobertweet-text-only-s1"
TEXT_S2 = "evelnap/indobertweet-text-only-s2"
META_S1 = "evelnap/indobertweet-metadata-s1"
META_S2 = "evelnap/indobertweet-metadata-s2"

THRESHOLD = 0.5
POLARIZED = 0        # dipakai kalau CSV tidak punya kolom polarized
BATCH_SIZE = 32
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

device: cuda


## Load data

In [27]:
META_COLS = [
    "ethnicity", "religion", "disability", "lgbt", "gender", "age", "domisili",
    "pendidikan terakhir", "status pekerjaan", "president vote leaning",
]
LABELS_S2 = [
    "insults", "identity_attack", "threat_incitement_to_violence",
    "profanity_obscenity", "sexually_explicit",
]
LABELS = ["toxicity"] + LABELS_S2


def text_only_prefix(pol_avg):
    return f"Nilai rata-rata polarisasi: {pol_avg:.2f}"


def metadata_string(row, with_polarized=True):
    meta = f"Polarisasi: {row['polarized']}. " if with_polarized else ""
    return meta + (
        f"Etnis: {row['ethnicity']}. "
        f"Agama: {row['religion']}. "
        f"Disabilitas: {row['disability']}. "
        f"LGBT: {row['lgbt']}. "
        f"Gender: {row['gender']}. "
        f"Usia: {row['age']}. "
        f"Domisili: {row['domisili']}. "
        f"Pendidikan: {row['pendidikan terakhir']}. "
        f"Pekerjaan: {row['status pekerjaan']}. "
        f"Preferensi politik: {row['president vote leaning']}."
    )


class Predictor:
    def __init__(self, repo):
        self.tokenizer = AutoTokenizer.from_pretrained(repo)
        self.model = AutoModelForSequenceClassification.from_pretrained(repo).to(DEVICE).eval()
        self.multi_label = self.model.config.problem_type == "multi_label_classification"
        print(f"Loaded {repo} ({'multi-label' if self.multi_label else 'single-label'})")

    def label_names(self, default):
        """Urutan label dari config model; fallback ke default kalau id2label generik (LABEL_0, ...)."""
        names = [self.model.config.id2label[i] for i in range(self.model.config.num_labels)]
        return names if set(names) == set(default) else default

    def predict(self, first, second=None, max_length=128):
        """Return prob kelas toxic (S1) atau prob per label (S2)."""
        out = []
        for i in range(0, len(first), BATCH_SIZE):
            a = first[i:i + BATCH_SIZE]
            b = second[i:i + BATCH_SIZE] if second is not None else None
            enc = self.tokenizer(
                a, b,
                truncation="only_second" if b is not None else True,
                max_length=max_length, padding="max_length", return_tensors="pt",
            ).to(DEVICE)
            with torch.no_grad():
                logits = self.model(**enc).logits
            probs = torch.sigmoid(logits) if self.multi_label else torch.softmax(logits, dim=-1)[:, 1]
            out.append(probs.cpu().numpy())
        return np.concatenate(out)

In [28]:
rows = pd.read_csv(CASES_PATH)
rows["processed_text"] = rows["text"].apply(preprocess_text)

profiles = None
if os.path.exists(PROFILES_PATH):
    df = pd.read_csv(PROFILES_PATH)
    df = df[(df["is_noise_or_spam_text"] == 0) & (df["is_valid"] == 1)].dropna()
    profiles = df[["annotator_id"] + META_COLS].drop_duplicates("annotator_id").set_index("annotator_id").sort_index()
    print(f"{len(profiles)} profil annotator dari data training")
else:
    print(f"[WARN] {PROFILES_PATH} tidak ada, metadata dipakai apa adanya dari CSV")

has_meta = "annotator_id" in rows.columns and set(META_COLS) <= set(rows.columns)
if USE_ALL_ANNOTATORS or not has_meta:
    assert profiles is not None, "butuh PROFILES_PATH untuk memasangkan teks dengan semua annotator"
    keep = [c for c in ["case_id", "text", "processed_text", "polarized"] if c in rows.columns]
    rows = rows.drop_duplicates("case_id")[keep].merge(profiles.reset_index(), how="cross")
elif profiles is not None:
    # samakan string metadata dengan data training, supaya input model sama persis
    known = rows["annotator_id"].isin(profiles.index)
    for col in META_COLS:
        rows.loc[known, col] = rows.loc[known, "annotator_id"].map(profiles[col])

if "polarized" not in rows.columns:
    rows["polarized"] = POLARIZED
rows = rows.reset_index(drop=True)
texts = rows.drop_duplicates("case_id")[["case_id", "text", "processed_text", "polarized"]].reset_index(drop=True)
print(f"{len(texts)} teks, {len(rows)} pasangan (teks, annotator)")

[WARN] ../data/explode_dataset_using_latest_cleaned_text.csv tidak ada, metadata dipakai apa adanya dari CSV
20 teks, 40 pasangan (teks, annotator)


In [29]:
# Contoh input ke model untuk baris pertama
row = rows.iloc[0]
print("[teks-only S1]", text_only_prefix(float(row["polarized"])), "||", row["processed_text"])
print("[metadata  S1]", metadata_string(row), "||", row["processed_text"])
print("[teks-only S2]", row["processed_text"])
print("[metadata  S2]", metadata_string(row, with_polarized=False), "||", row["processed_text"])

[teks-only S1] Nilai rata-rata polarisasi: 0.00 || azan subuh pakai toa keras banget, ganggu orang tidur. ibadah kok maksa tetangga ikut dengar.
[metadata  S1] Polarisasi: 0. Etnis: Sunda. Agama: Islam. Disabilitas: no. LGBT: no. Gender: M. Usia: 43. Domisili: Jakarta. Pendidikan: Magister (S2). Pekerjaan: Bekerja. Preferensi politik: 2. || azan subuh pakai toa keras banget, ganggu orang tidur. ibadah kok maksa tetangga ikut dengar.
[teks-only S2] azan subuh pakai toa keras banget, ganggu orang tidur. ibadah kok maksa tetangga ikut dengar.
[metadata  S2] Etnis: Sunda. Agama: Islam. Disabilitas: no. LGBT: no. Gender: M. Usia: 43. Domisili: Jakarta. Pendidikan: Magister (S2). Pekerjaan: Bekerja. Preferensi politik: 2. || azan subuh pakai toa keras banget, ganggu orang tidur. ibadah kok maksa tetangga ikut dengar.


## Prediksi

Model teks-only cukup sekali per teks; model metadata sekali per (teks, annotator).


In [30]:
# Stage 1: toxicity
m = Predictor(TEXT_S1)
prefix = [text_only_prefix(float(p)) for p in texts["polarized"]]
texts["text_toxicity"] = m.predict(prefix, texts["processed_text"].tolist(), max_length=128)

m = Predictor(META_S1)
meta = rows.apply(metadata_string, axis=1).tolist()
rows["meta_toxicity"] = m.predict(meta, rows["processed_text"].tolist(), max_length=256)

# Stage 2: jenis toxicity
m = Predictor(TEXT_S2)
probs = m.predict(texts["processed_text"].tolist(), max_length=128)
for i, c in enumerate(m.label_names(LABELS_S2)):
    texts[f"text_{c}"] = probs[:, i]

m = Predictor(META_S2)
meta = rows.apply(metadata_string, axis=1, with_polarized=False).tolist()
probs = m.predict(meta, rows["processed_text"].tolist(), max_length=256)
for i, c in enumerate(m.label_names(LABELS_S2)):
    rows[f"meta_{c}"] = probs[:, i]

del m
torch.cuda.empty_cache()
rows = rows.merge(texts[["case_id"] + [f"text_{c}" for c in LABELS]], on="case_id")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loaded evelnap/indobertweet-text-only-s1 (single-label)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loaded evelnap/indobertweet-metadata-s1 (single-label)


config.json:   0%|          | 0.00/1.28k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/379 [00:00<?, ?B/s]

... (8 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B /  442MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loaded evelnap/indobertweet-metadata-s2 (multi-label)


## Bandingkan

Satu baris per (teks, annotator, label).


In [31]:
parts = []
for lab in LABELS:
    d = rows[["case_id", "annotator_id", "text"]].copy()
    d["label"] = lab
    d["text_prob"] = rows[f"text_{lab}"]
    d["meta_prob"] = rows[f"meta_{lab}"]
    parts.append(d)
long = pd.concat(parts, ignore_index=True)
long["text_pred"] = (long["text_prob"] >= THRESHOLD).astype(int)
long["meta_pred"] = (long["meta_prob"] >= THRESHOLD).astype(int)
long["sama_dengan_text_only"] = long["text_pred"] == long["meta_pred"]
long["selisih_prob"] = (long["meta_prob"] - long["text_prob"]).round(4)

## Lihat contoh teks

Sel pertama: daftar semua teks beserta pola toxicity-nya.

Sel kedua: per TC, teksnya beserta hasil model teks-only dan hasil model metadata untuk tiap annotator: **toxic atau tidak**, dan kalau toxic **masuk label apa** (label stage 2 dengan prob >= `THRESHOLD`). Atur `SHOW_CASES` untuk memilih TC tertentu.


In [32]:
g = long.groupby(["case_id", "label"], sort=False)
per_text = g.agg(
    text=("text", "first"),
    text_prob=("text_prob", "first"),
    text_pred=("text_pred", "first"),
    meta_prob_min=("meta_prob", "min"),
    meta_prob_max=("meta_prob", "max"),
    n_annotator=("meta_pred", "size"),
    n_meta_toxic=("meta_pred", "sum"),
    n_sama_text_only=("sama_dengan_text_only", "sum"),
).reset_index()
per_text["meta_prob_range"] = per_text["meta_prob_max"] - per_text["meta_prob_min"]
varies = (per_text["n_meta_toxic"] > 0) & (per_text["n_meta_toxic"] < per_text["n_annotator"])
per_text["pola"] = np.select(
    [varies, per_text["n_sama_text_only"] == per_text["n_annotator"]],
    ["bergantung metadata", "ikut text-only"],
    "beda dari text-only, seragam",
)
per_text = per_text.round(4)
per_text[per_text["label"] == "toxicity"].drop(columns="label")

,case_id,text,text_prob,text_pred,meta_prob_min,meta_prob_max,n_annotator,n_meta_toxic,n_sama_text_only,meta_prob_range,pola
0,TC01,"Azan subuh pakai toa keras banget, ganggu oran...",0.0979,0,0.0121,0.1036,2,0,2,0.0915,ikut text-only
1,TC02,"Natal itu perayaan kafir, haram hukumnya ucapi...",0.5336,1,0.0153,0.9854,2,1,1,0.9701,bergantung metadata
2,TC03,"Pastor-pastor itu nggak boleh nikah, pantes aj...",0.0670,0,0.0061,0.1546,2,0,2,0.1485,ikut text-only
3,TC04,Kok masih ada ya orang nyembah patung sama sap...,0.2179,0,0.0206,0.8955,2,1,1,0.8749,bergantung metadata
4,TC05,"Biksu di Myanmar itu munafik, ngajarin welas a...",0.8525,1,0.3568,0.9883,2,1,1,0.6314,bergantung metadata
5,TC06,"Syiah itu bukan Islam, Rafidhah pencaci sahaba...",0.6817,1,0.0744,0.9907,2,1,1,0.9163,bergantung metadata
6,TC07,"Ahmadiyah silakan beribadah, tapi jangan pakai...",0.0250,0,0.0047,0.2669,2,0,2,0.2622,ikut text-only
7,TC08,Penghayat kepercayaan itu isinya cuma klenik s...,0.0791,0,0.4758,0.7137,2,1,1,0.2379,bergantung metadata
8,TC09,"Bos saya Cina tapi baik banget loh, jarang-jar...",0.0214,0,0.0208,0.0418,2,0,2,0.0210,ikut text-only
9,TC10,"Mantap, pemerintah akhirnya usir investor Cina...",0.0104,0,0.0629,0.0983,2,0,2,0.0354,ikut text-only


In [33]:
# Detail toxicity per (teks, annotator)
show = rows[["case_id", "annotator_id", "religion", "ethnicity", "gender", "president vote leaning"]].copy()
show["text_prob"] = rows["text_toxicity"].round(4)
show["meta_prob"] = rows["meta_toxicity"].round(4)
show["text_pred"] = (rows["text_toxicity"] >= THRESHOLD).astype(int)
show["meta_pred"] = (rows["meta_toxicity"] >= THRESHOLD).astype(int)
if "toxicity" in rows.columns:          # label ekspektasi dari CSV, kalau ada
    show["ekspektasi"] = rows["toxicity"]
show

,case_id,annotator_id,religion,ethnicity,gender,president vote leaning,text_prob,meta_prob,text_pred,meta_pred,ekspektasi
0,TC01,20,Islam,Sunda,M,2,0.0979,0.1036,0,0,1
1,TC01,19,Kristen,Batak,F,2,0.0979,0.0121,0,0,0
2,TC02,1,Kristen,Tionghoa,F,2,0.5336,0.0153,1,0,1
3,TC02,23,Islam,Ternate,M,1,0.5336,0.9854,1,1,0
4,TC03,13,Katolik,Dayak,M,2,0.0670,0.1546,0,0,1
5,TC03,31,Islam,Tobelo,M,1,0.0670,0.0061,0,0,0
6,TC04,18,Hindu,Bali,M,2,0.2179,0.8955,0,1,1
7,TC04,30,Islam,Minang,F,1,0.2179,0.0206,0,0,0
8,TC05,10,Buddha,Tionghoa,F,3,0.8525,0.3568,1,0,1
9,TC05,25,Islam,Aceh,F,3,0.8525,0.9883,1,1,0


In [34]:
summary = per_text.groupby("label", sort=False).agg(
    n_teks=("case_id", "size"),
    ikut_text_only=("pola", lambda s: (s == "ikut text-only").sum()),
    beda_seragam=("pola", lambda s: (s == "beda dari text-only, seragam").sum()),
    bergantung_metadata=("pola", lambda s: (s == "bergantung metadata").sum()),
    rata2_meta_prob_range=("meta_prob_range", "mean"),
)
summary["pct_pasangan_sama_text_only"] = long.groupby("label", sort=False)["sama_dengan_text_only"].mean()
summary.round(4)

,n_teks,ikut_text_only,beda_seragam,bergantung_metadata,rata2_meta_prob_range,pct_pasangan_sama_text_only
label,,,,,,
toxicity,20,10,0,10,0.4509,0.750
insults,20,11,5,4,0.1815,0.650
identity_attack,20,10,6,4,0.2330,0.600
threat_incitement_to_violence,20,18,0,2,0.1205,0.950
profanity_obscenity,20,14,3,3,0.1571,0.775
sexually_explicit,20,18,0,2,0.0914,0.950


In [35]:
# Opsional: kalau CSV punya label ekspektasi per annotator (kolom toxicity), model mana yang lebih cocok?
if "ekspektasi" in show.columns:
    acc_text = (show["text_pred"] == show["ekspektasi"]).mean()
    acc_meta = (show["meta_pred"] == show["ekspektasi"]).mean()
    print(f"Akurasi toxicity vs ekspektasi -> teks-only: {acc_text:.4f} | metadata: {acc_meta:.4f}")

Akurasi toxicity vs ekspektasi -> teks-only: 0.5000 | metadata: 0.4500


In [36]:
os.makedirs(OUT_DIR, exist_ok=True)
long.to_csv(f"{OUT_DIR}/prediksi_per_annotator.csv", index=False)
per_text.to_csv(f"{OUT_DIR}/ringkasan_per_teks.csv", index=False)
summary.round(4).to_csv(f"{OUT_DIR}/ringkasan_per_label.csv")
print(f"Hasil tersimpan di {OUT_DIR}/")

Hasil tersimpan di results/


In [39]:
# Daftar semua teks + pola toxicity
pd.set_option("display.max_colwidth", None)
per_text[per_text["label"] == "toxicity"][
    ["case_id", "text", "text_prob", "text_pred", "n_meta_toxic", "n_annotator", "meta_prob_range", "pola"]
]

,case_id,text,text_prob,text_pred,n_meta_toxic,n_annotator,meta_prob_range,pola
0,TC01,"Azan subuh pakai toa keras banget, ganggu orang tidur. Ibadah kok maksa tetangga ikut dengar.",0.0979,0,0,2,0.0915,ikut text-only
1,TC02,"Natal itu perayaan kafir, haram hukumnya ucapin selamat, nanti ikut masuk neraka bareng mereka.",0.5336,1,1,2,0.9701,bergantung metadata
2,TC03,"Pastor-pastor itu nggak boleh nikah, pantes aja banyak skandal di gereja.",0.0670,0,0,2,0.1485,ikut text-only
3,TC04,Kok masih ada ya orang nyembah patung sama sapi di zaman modern begini 🤣,0.2179,0,1,2,0.8749,bergantung metadata
4,TC05,"Biksu di Myanmar itu munafik, ngajarin welas asih tapi diam saja lihat Rohingya dibantai.",0.8525,1,1,2,0.6314,bergantung metadata
5,TC06,"Syiah itu bukan Islam, Rafidhah pencaci sahabat Nabi, umat wajib waspada.",0.6817,1,1,2,0.9163,bergantung metadata
6,TC07,"Ahmadiyah silakan beribadah, tapi jangan pakai nama Islam. Bikin agama sendiri saja biar umat tidak bingung.",0.0250,0,0,2,0.2622,ikut text-only
7,TC08,"Penghayat kepercayaan itu isinya cuma klenik sama sesajen, kolom agama di KTP nggak usah diisi aneh-aneh.",0.0791,0,1,2,0.2379,bergantung metadata
8,TC09,"Bos saya Cina tapi baik banget loh, jarang-jarang ada yang kayak gini.",0.0214,0,0,2,0.0210,ikut text-only
9,TC10,"Mantap, pemerintah akhirnya usir investor Cina. Tanah air harus untuk pribumi!",0.0104,0,0,2,0.0354,ikut text-only


In [40]:
SHOW_CASES = None   # None = semua TC, atau list mis. ["TC01", "TC05"]


def hasil(prob):
    """prob: dict label -> prob. Return 'TOXIC -> label, ...' atau 'NON-TOXIC'."""
    if prob["toxicity"] < THRESHOLD:
        return f"NON-TOXIC ({prob['toxicity']:.2f})"
    labels = [l for l in LABELS_S2 if prob[l] >= THRESHOLD]
    if not labels:   # toxic tapi tidak ada label stage 2 yang lewat threshold -> tampilkan yang tertinggi
        top = max(LABELS_S2, key=lambda l: prob[l])
        return f"TOXIC ({prob['toxicity']:.2f}) -> tidak ada label >= {THRESHOLD} (tertinggi: {top} {prob[top]:.2f})"
    return f"TOXIC ({prob['toxicity']:.2f}) -> " + ", ".join(f"{l} ({prob[l]:.2f})" for l in labels)


def profil(r):
    p = f"{r['religion']}, {r['ethnicity']}, {str(r['gender']).strip()}, {r['age']} th, pilpres {r['president vote leaning']}"
    if r["disability"] == "yes":
        p += ", disabilitas"
    if r["lgbt"] == "yes":
        p += ", LGBT"
    return p


for _, t in texts.iterrows():
    if SHOW_CASES and t["case_id"] not in SHOW_CASES:
        continue
    print("=" * 100)
    print(f"[{t['case_id']}] {t['text']}")
    print("-" * 100)
    print(f"Teks-only : {hasil({l: t[f'text_{l}'] for l in LABELS})}")
    print("Metadata  :")
    for _, r in rows[rows["case_id"] == t["case_id"]].iterrows():
        eksp = f"   [ekspektasi: {'toxic' if r['toxicity'] == 1 else 'non-toxic'}]" if "toxicity" in rows.columns else ""
        print(f"  - annotator {r['annotator_id']:>2} ({profil(r)})")
        print(f"      {hasil({l: r[f'meta_{l}'] for l in LABELS})}{eksp}")
    print()

[TC01] Azan subuh pakai toa keras banget, ganggu orang tidur. Ibadah kok maksa tetangga ikut dengar.
----------------------------------------------------------------------------------------------------
Teks-only : NON-TOXIC (0.10)
Metadata  :
  - annotator 20 (Islam, Sunda, M, 43 th, pilpres 2)
      NON-TOXIC (0.10)   [ekspektasi: toxic]
  - annotator 19 (Kristen, Batak, F, 39 th, pilpres 2)
      NON-TOXIC (0.01)   [ekspektasi: non-toxic]

[TC02] Natal itu perayaan kafir, haram hukumnya ucapin selamat, nanti ikut masuk neraka bareng mereka.
----------------------------------------------------------------------------------------------------
Teks-only : TOXIC (0.53) -> identity_attack (0.67)
Metadata  :
  - annotator  1 (Kristen, Tionghoa, F, 50 th, pilpres 2)
      NON-TOXIC (0.02)   [ekspektasi: toxic]
  - annotator 23 (Islam, Ternate, M, 35 th, pilpres 1)
      TOXIC (0.99) -> insults (0.95)   [ekspektasi: non-toxic]

[TC03] Pastor-pastor itu nggak boleh nikah, pantes aja banyak ska

Jawaban singkatnya: hasil ini cukup kuat untuk klaim "metadata berpengaruh", tapi belum cukup untuk klaim "model menangkap perspektif minoritas". Dan banyaknya hasil yang tidak sesuai ekspektasi ternyata ada penjelasannya.

1. Metadata memang berpengaruh, ini terbukti
Teksnya sama persis, cuma annotator-nya yang beda:

10 dari 20 teks labelnya berbeda antara dua annotator (TC02, 04, 05, 06, 08, 16, 17, 18, 19, 20).
Selisih probabilitasnya ekstrem. Contohnya TC02 (0.02 vs 0.99), TC16 (0.03 vs 0.96), dan TC18 (0.03 vs 0.98).
Jadi model metadata tidak sekadar mengikuti pola teks-only (majority vote). Untuk klaim ini, hasilnya sudah kuat.

2. Tapi yang ditangkap ternyata "ketatnya annotator", bukan identitas kelompoknya
Dari 10 teks yang labelnya berbeda, hanya 4 yang sesuai ekspektasi; 6 lainnya terbalik. Saya bandingkan dengan toxicity rate tiap annotator di data training, yaitu persentase teks yang dia beri label toxic. Hasilnya 10 dari 10 cocok: annotator yang secara umum lebih sering memberi label toxic selalu diprediksi toxic.

Case	Diprediksi toxic	Diprediksi non-toxic
TC02 (ucapan Natal)	#23 Islam, rate 0.44	#1 Kristen, rate 0.06
TC16 ("bencong")	#11 LGBT, rate 0.24	#27 Aceh, rate 0.02
TC18 ("boomer")	#16 umur 19, rate 0.31	#27 umur 59, rate 0.02
Artinya, model belajar "annotator ini galak, annotator itu longgar". Ia belum terlihat belajar "orang Kristen lebih tersinggung kalau Natal dihina". Ini masuk akal secara teknis: ke-32 annotator masing-masing punya kombinasi metadata yang unik, jadi string metadata bisa berfungsi seperti ID annotator. Model cukup menghafal kecenderungan per orang.

3. Kenapa banyak yang "tidak expected"
Label ekspektasinya hipotesis saya, bukan ground truth. Hipotesis yang salah bukan berarti modelnya salah.
Hanya 1 annotator per kelompok. Sinyal dari kelompoknya (misalnya Kristen) jadi tidak bisa dipisahkan dari kebiasaan pribadi orang itu.
Teksnya sintetis dan cuma 20. Ini terlalu sedikit untuk klaim statistik.
Cara membingkai di tesis
Model metadata menghasilkan prediksi yang berbeda untuk teks yang sama bergantung pada annotator (10/20 teks berubah label), sedangkan model teks-only hanya merepresentasikan pola mayoritas. Namun analisis lanjut menunjukkan perbedaan ini konsisten dengan tingkat keketatan masing-masing annotator (10/10), sehingga model lebih menangkap kecenderungan individual annotator daripada sensitivitas berbasis identitas kelompok.

Ini temuan yang jujur dan tetap menarik. Faktanya model memang "mendengar" annotator individual, yang tidak bisa dilakukan model majority vote.

Supaya argumennya lebih kuat
Ada tes yang bisa langsung memisahkan "identitas" dari "keketatan annotator": tes counterfactual. Annotator-nya tetap, lalu satu atribut saja diganti, misalnya agama Islam diganti Kristen dengan atribut lain tetap. Kalau probabilitasnya berubah sesuai topik teks, berarti atribut itu memang berpengaruh. Kalau tidak berubah, model memang hanya menghafal profil annotator.